# 4D-Var Data Assimilation with the Shallow Water Model

This notebook demonstrates **4D-Var data assimilation** — recovering the initial conditions
of a shallow water flow from sparse, noisy observations at later times.

## What is 4D-Var?

4D-Var (four-dimensional variational data assimilation) finds the optimal initial state
$x_0$ by minimizing a cost function:

$$J(x_0) = \frac{1}{2}(x_0 - x^b)^T B^{-1}(x_0 - x^b) + \frac{1}{2}\sum_k \|H[M(x_0, t_k)] - y_k\|_{R^{-1}}^2$$

where:
- $x^b$ is a background (prior) estimate, $B$ its error covariance
- $M(x_0, t_k)$ is the model forecast from $x_0$ to time $t_k$
- $H$ is the observation operator (extracts observed locations from model state)
- $y_k$ are observations at time $t_k$, $R$ their error covariance

The gradient $\nabla J$ is computed efficiently via **reverse-mode AD** (the adjoint),
which JAX provides automatically through `jax.grad`.

## Twin Experiment Setup

We use a "twin experiment" protocol:
1. Run the model from a known "truth" initial condition
2. Sample **sparse**, noisy observations from this truth run (only 50% of grid points)
3. Start from a wrong initial condition (the "background")
4. Use 4D-Var to recover the true initial condition

The key power of 4D-Var is that it uses the **model dynamics** to propagate information
from sparse observations across space and time — even grid points that are never directly
observed can be corrected through dynamical coupling.

## Outline

1. Set up the model and generate the truth run
2. Create sparse, noisy observations
3. Define the 4D-Var cost function
4. Run the optimization with L-BFGS
5. Visualize the recovery

## 1. Imports and Setup

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize

jax.config.update("jax_enable_x64", True)

from initial_conditions import initialize_interior
from swm_array_api import timestep, interior_to_halo

print(f"JAX version: {jax.__version__}")
print(f"Default device: {jax.devices()[0]}")

In [ ]:
# Model parameters
M, N = 16, 16
dx = 100000.0
dy = 100000.0
dt = 90.0
a = 1000000.0
alpha = 0.001

xp = jnp

## 2. Forward Model

We wrap the SWM timestep in `jax.lax.scan` for efficient AD.
The model returns intermediate states at observation times.

In [ ]:
def initial_state(u_interior, v_interior, p_interior):
    """Build full state with halos from interior fields."""
    u = interior_to_halo(xp, u_interior)
    v = interior_to_halo(xp, v_interior)
    p = interior_to_halo(xp, p_interior)
    return (u, v, p, u_interior, v_interior, p_interior)


def forward_model(u_int, v_int, p_int, n_steps):
    """Run the SWM forward for n_steps timesteps.

    Returns the final state (u, v, p) with halos.
    """
    state = initial_state(u_int, v_int, p_int)
    u, v, p, uold, vold, pold = state

    # First step: forward Euler
    unew, vnew, pnew, uold, vold, pold = timestep(
        xp, u, v, p, uold, vold, pold, dx, dy, dt, 0.0, M, N
    )

    # Remaining steps: leapfrog via scan
    carry = (unew, vnew, pnew, uold, vold, pold)

    def scan_step(carry, _):
        u, v, p, uold, vold, pold = carry
        unew, vnew, pnew, uold_new, vold_new, pold_new = timestep(
            xp, u, v, p, uold, vold, pold, dx, dy, 2.0 * dt, alpha, M, N
        )
        return (unew, vnew, pnew, uold_new, vold_new, pold_new), None

    final_carry, _ = jax.lax.scan(scan_step, carry, None, length=n_steps - 1)
    return final_carry[0], final_carry[1], final_carry[2]


def forward_model_with_snapshots(u_int, v_int, p_int, n_steps, obs_steps):
    """Run forward model, saving interior snapshots at specified timesteps.

    Args:
        u_int, v_int, p_int: Initial interior fields (M, N).
        n_steps: Total timesteps.
        obs_steps: Sorted array of timestep indices at which to save snapshots.

    Returns:
        snapshots_u, snapshots_v, snapshots_p: Arrays of shape (n_obs, M, N).
    """
    state = initial_state(u_int, v_int, p_int)
    u, v, p, uold, vold, pold = state

    # First step
    unew, vnew, pnew, uold, vold, pold = timestep(
        xp, u, v, p, uold, vold, pold, dx, dy, dt, 0.0, M, N
    )

    carry = (unew, vnew, pnew, uold, vold, pold)

    # Use scan over all steps, saving interior state at each step
    def scan_step_save(carry, _):
        u, v, p, uold, vold, pold = carry
        unew, vnew, pnew, uold_new, vold_new, pold_new = timestep(
            xp, u, v, p, uold, vold, pold, dx, dy, 2.0 * dt, alpha, M, N
        )
        new_carry = (unew, vnew, pnew, uold_new, vold_new, pold_new)
        # Save interior of current step's output
        snapshot = (unew[1:-1, 1:-1], vnew[1:-1, 1:-1], pnew[1:-1, 1:-1])
        return new_carry, snapshot

    final_carry, all_snapshots = jax.lax.scan(scan_step_save, carry, None, length=n_steps - 1)
    # all_snapshots: (u_snaps, v_snaps, p_snaps) each of shape (n_steps-1, M, N)

    # Also include the first step output (index 0 in the scan corresponds to step 1)
    first_snap = (
        unew[1:-1, 1:-1][None, :, :],
        vnew[1:-1, 1:-1][None, :, :],
        pnew[1:-1, 1:-1][None, :, :],
    )
    u_all = jnp.concatenate([first_snap[0], all_snapshots[0]], axis=0)  # (n_steps, M, N)
    v_all = jnp.concatenate([first_snap[1], all_snapshots[1]], axis=0)
    p_all = jnp.concatenate([first_snap[2], all_snapshots[2]], axis=0)

    # Extract only at observation times (obs_steps are 1-based timestep indices)
    # Step index i in the concatenated array corresponds to timestep i+1
    # (step 0 is the first Euler step result = timestep 1)
    obs_indices = jnp.array(obs_steps) - 1  # convert to 0-based
    return u_all[obs_indices], v_all[obs_indices], p_all[obs_indices]


print("Forward model functions defined.")

## 3. Generate the Truth and Observations

We use the default sinusoidal stream function initialization as our "truth" initial
condition. The background (wrong guess) is a strongly perturbed version (50% of field std).

We observe **all three fields** (u, v, p) at a **random 50% subset** of grid points.
This sparse observation network tests 4D-Var's ability to use model dynamics to
propagate information from observed to unobserved locations.

The assimilation window spans **100 timesteps** (2.5 hours), long enough for the dynamics
to propagate information across the domain while keeping the tangent linear approximation
reasonably valid.

In [ ]:
# Assimilation window
N_STEPS = 100

# Observation times (timestep indices within the window)
obs_steps = np.array([20, 40, 60, 80, 100])
n_obs_times = len(obs_steps)

# Sparse observation network: observe 50% of grid points
obs_fraction = 0.50
n_obs_points = int(obs_fraction * M * N)

# Background perturbation: 50% of field std — large enough for a dramatic demo
bg_perturbation = 0.5

# Observation noise: 5% of field variability
obs_noise_frac = 0.05

key = jax.random.PRNGKey(42)

# Generate random observation mask (same locations at all times)
key, subkey = jax.random.split(key)
obs_indices = jax.random.choice(subkey, M * N, shape=(n_obs_points,), replace=False)
obs_indices = jnp.sort(obs_indices)

print(f"Assimilation window: {N_STEPS} steps ({N_STEPS * dt / 3600:.1f} hours)")
print(f"Observation times: {obs_steps} (steps)")
print(f"Observed grid points: {n_obs_points} out of {M * N} ({obs_fraction * 100:.0f}%)")
print(f"Background perturbation: {bg_perturbation * 100:.0f}% of field std")
print(f"Observation noise: {obs_noise_frac * 100:.0f}% of field variability")

In [ ]:
# Truth initial condition
u_true_int, v_true_int, p_true_int = initialize_interior(xp, M, N, dx, dy, a)

# Run truth forward and save snapshots at observation times
print("Running truth simulation...")
u_true_snaps, v_true_snaps, p_true_snaps = forward_model_with_snapshots(
    u_true_int, v_true_int, p_true_int, N_STEPS, obs_steps
)
p_true_snaps.block_until_ready()
print(f"Truth snapshots shape: {p_true_snaps.shape}")

# Compute field variability for noise scaling
u_std = float(jnp.std(u_true_snaps))
v_std = float(jnp.std(v_true_snaps))
p_std = float(jnp.std(p_true_snaps))
print(f"Field std: u={u_std:.4f}, v={v_std:.4f}, p={p_std:.4f}")

In [ ]:
# Generate noisy observations from truth at sparse locations
# Noise is 5% of each field's variability.

sigma_o_u = obs_noise_frac * u_std
sigma_o_v = obs_noise_frac * v_std
sigma_o_p = obs_noise_frac * p_std

key, *subkeys = jax.random.split(key, 4)

# Extract observations at sparse locations only
u_obs = u_true_snaps.reshape(n_obs_times, -1)[:, obs_indices] + sigma_o_u * jax.random.normal(
    subkeys[0], (n_obs_times, n_obs_points)
)
v_obs = v_true_snaps.reshape(n_obs_times, -1)[:, obs_indices] + sigma_o_v * jax.random.normal(
    subkeys[1], (n_obs_times, n_obs_points)
)
p_obs = p_true_snaps.reshape(n_obs_times, -1)[:, obs_indices] + sigma_o_p * jax.random.normal(
    subkeys[2], (n_obs_times, n_obs_points)
)

print(f"Observations generated (all three fields at {n_obs_points} sparse locations):")
print(f"  Shape per field: {p_obs.shape} (n_times x n_obs_points)")
print(f"  Noise std: u={sigma_o_u:.4f}, v={sigma_o_v:.4f}, p={sigma_o_p:.4f}")

In [ ]:
# Visualize observation locations
obs_mask = np.zeros(M * N)
obs_mask[np.array(obs_indices)] = 1.0
obs_mask_2d = obs_mask.reshape(M, N)

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.imshow(obs_mask_2d, cmap="Greys", origin="lower", interpolation="nearest")
ax.set_title(
    f"Observation locations ({n_obs_points}/{M * N} = {obs_fraction * 100:.0f}%)", fontsize=13
)
ax.set_xlabel("Grid x")
ax.set_ylabel("Grid y")
plt.tight_layout()
plt.show()

In [ ]:
# Background (wrong) initial condition: truth + 50% perturbation
key, *subkeys = jax.random.split(key, 4)

sigma_b_u = bg_perturbation * float(jnp.std(u_true_int))
sigma_b_v = bg_perturbation * float(jnp.std(v_true_int))
sigma_b_p = bg_perturbation * float(jnp.std(p_true_int))

u_bg_int = u_true_int + sigma_b_u * jax.random.normal(subkeys[0], u_true_int.shape)
v_bg_int = v_true_int + sigma_b_v * jax.random.normal(subkeys[1], v_true_int.shape)
p_bg_int = p_true_int + sigma_b_p * jax.random.normal(subkeys[2], p_true_int.shape)

print(
    f"Background error covariance (sigma_b): u={sigma_b_u:.4f}, v={sigma_b_v:.4f}, p={sigma_b_p:.4f}"
)
print(f"Actual background RMS errors:")
for name, t, b in [
    ("u", u_true_int, u_bg_int),
    ("v", v_true_int, v_bg_int),
    ("p", p_true_int, p_bg_int),
]:
    rms = float(jnp.sqrt(jnp.mean((b - t) ** 2)))
    print(f"  {name}: {rms:.4f}")

## 4. Visualize the Setup

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Truth initial p
im0 = axes[0].imshow(np.array(p_true_int), cmap="viridis", origin="lower")
axes[0].set_title("Truth initial p", fontsize=13)
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# Background initial p
im1 = axes[1].imshow(np.array(p_bg_int), cmap="viridis", origin="lower")
axes[1].set_title("Background initial p", fontsize=13)
plt.colorbar(im1, ax=axes[1], shrink=0.8)

# Difference
diff = np.array(p_bg_int - p_true_int)
vmax = np.abs(diff).max()
im2 = axes[2].imshow(diff, cmap="RdBu_r", vmin=-vmax, vmax=vmax, origin="lower")
axes[2].set_title("Background - Truth", fontsize=13)
plt.colorbar(im2, ax=axes[2], shrink=0.8)

plt.suptitle("Initial Conditions for 4D-Var (pressure field)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize background errors for all fields
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, (t, b, name) in zip(
    axes,
    [
        (u_true_int, u_bg_int, "u"),
        (v_true_int, v_bg_int, "v"),
        (p_true_int, p_bg_int, "p"),
    ],
):
    diff = np.array(b - t)
    vmax = np.abs(diff).max()
    im = ax.imshow(diff, cmap="RdBu_r", vmin=-vmax, vmax=vmax, origin="lower")
    rms = np.sqrt(np.mean(diff**2))
    ax.set_title(f"Background error in {name}₀ (RMS={rms:.4f})", fontsize=13)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle("Background errors (background - truth)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Define the 4D-Var Cost Function

The cost has two terms:

1. **Background term** $J_b$: penalizes deviation from the background (regularization)
2. **Observation term** $J_o$: penalizes mismatch between model forecast and observations

$$J(x_0) = \frac{1}{2}\sum_f \left\|\frac{x_0^f - x_b^f}{\sigma_b^f}\right\|^2 + \frac{1}{2}\sum_{k,f} \left\|\frac{H[M^f(x_0, t_k)] - y_k^f}{\sigma_o^f}\right\|^2$$

where $f$ indexes the fields (u, v, p), $k$ indexes the observation times,
and $H$ selects the sparse observed grid points.
We use diagonal error covariances (scalar $\sigma$ per field).

In [ ]:
def cost_4dvar(u_int, v_int, p_int):
    """4D-Var cost function.

    Args:
        u_int, v_int, p_int: Trial initial condition (interior, M x N).

    Returns:
        Scalar cost J = J_b + J_o.
    """
    # Background term: J_b = 0.5 * sum((x0 - xb)^2 / sigma_b^2)
    J_b = (
        0.5 * jnp.sum(((u_int - u_bg_int) / sigma_b_u) ** 2)
        + 0.5 * jnp.sum(((v_int - v_bg_int) / sigma_b_v) ** 2)
        + 0.5 * jnp.sum(((p_int - p_bg_int) / sigma_b_p) ** 2)
    )

    # Observation term: run forward model and compare at sparse locations
    u_snaps, v_snaps, p_snaps = forward_model_with_snapshots(
        u_int, v_int, p_int, N_STEPS, obs_steps
    )

    # Extract model equivalents at observed locations
    u_model_obs = u_snaps.reshape(n_obs_times, -1)[:, obs_indices]
    v_model_obs = v_snaps.reshape(n_obs_times, -1)[:, obs_indices]
    p_model_obs = p_snaps.reshape(n_obs_times, -1)[:, obs_indices]

    J_o = (
        0.5 * jnp.sum(((u_model_obs - u_obs) / sigma_o_u) ** 2)
        + 0.5 * jnp.sum(((v_model_obs - v_obs) / sigma_o_v) ** 2)
        + 0.5 * jnp.sum(((p_model_obs - p_obs) / sigma_o_p) ** 2)
    )

    return J_b + J_o


# Verify cost at different points
J_at_truth = cost_4dvar(u_true_int, v_true_int, p_true_int)
J_at_bg = cost_4dvar(u_bg_int, v_bg_int, p_bg_int)

print(f"Cost at truth:      {float(J_at_truth):.4e}")
print(f"Cost at background: {float(J_at_bg):.4e}")
print(f"Ratio: {float(J_at_bg / J_at_truth):.0f}x (background is much worse)")

## 6. Gradient Computation and L-BFGS Optimization

We use `scipy.optimize.minimize` with `method='L-BFGS-B'`, providing JAX-computed
gradients. The control vector is flattened into a 1D array for scipy compatibility.

In [ ]:
# Create JIT-compiled cost and gradient function
cost_and_grad_fn = jax.jit(jax.value_and_grad(cost_4dvar, argnums=(0, 1, 2)))

# Warm up
print("JIT compiling cost + gradient (this may take a moment)...")
from time import perf_counter

t0 = perf_counter()
_val, _grads = cost_and_grad_fn(u_bg_int, v_bg_int, p_bg_int)
_grads[2].block_until_ready()
t_compile = perf_counter() - t0
print(f"Compile + first eval: {t_compile:.2f}s")

t0 = perf_counter()
_val, _grads = cost_and_grad_fn(u_bg_int, v_bg_int, p_bg_int)
_grads[2].block_until_ready()
t_exec = perf_counter() - t0
print(f"Subsequent eval:      {t_exec:.4f}s")

In [ ]:
# Pack/unpack utilities for scipy interface
def pack(u_int, v_int, p_int):
    """Pack three (M, N) arrays into a single 1D numpy array."""
    return np.concatenate(
        [
            np.array(u_int).ravel(),
            np.array(v_int).ravel(),
            np.array(p_int).ravel(),
        ]
    )


def unpack(x_flat):
    """Unpack a 1D array into three (M, N) jax arrays."""
    n = M * N
    u_int = jnp.array(x_flat[:n].reshape(M, N))
    v_int = jnp.array(x_flat[n : 2 * n].reshape(M, N))
    p_int = jnp.array(x_flat[2 * n :].reshape(M, N))
    return u_int, v_int, p_int


# Track optimization progress
cost_history = []
grad_norm_history = []
n_evals = [0]


def scipy_cost_and_grad(x_flat):
    """Wrapper for scipy: takes and returns numpy arrays."""
    u_int, v_int, p_int = unpack(x_flat)
    val, (gu, gv, gp) = cost_and_grad_fn(u_int, v_int, p_int)

    # Wait for computation
    val_float = float(val)

    # Pack gradient
    grad_flat = pack(gu, gv, gp)

    # Record progress
    cost_history.append(val_float)
    grad_norm_history.append(float(np.linalg.norm(grad_flat)))
    n_evals[0] += 1

    if n_evals[0] % 5 == 1:
        print(f"  eval {n_evals[0]:3d}: J = {val_float:.6e}, |∇J| = {grad_norm_history[-1]:.6e}")

    return val_float, grad_flat


print("Optimization interface ready.")
print(f"Control vector size: {3 * M * N} (3 fields x {M} x {N})")

In [ ]:
# Run L-BFGS-B optimization
print("Starting L-BFGS-B optimization...")
print("=" * 70)

x0 = pack(u_bg_int, v_bg_int, p_bg_int)  # start from the background

t0 = perf_counter()
result = minimize(
    scipy_cost_and_grad,
    x0,
    method="L-BFGS-B",
    jac=True,  # our function returns (value, gradient)
    options={
        "maxiter": 500,
        "ftol": 1e-14,
        "gtol": 1e-10,
    },
)
t_opt = perf_counter() - t0

print("=" * 70)
print(f"Optimization finished in {t_opt:.2f}s")
print(f"  Success: {result.success}")
print(f"  Message: {result.message}")
print(f"  Iterations: {result.nit}")
print(f"  Function evaluations: {result.nfev}")
print(f"  Final cost: {result.fun:.6e}")

In [ ]:
# Unpack the analysis (recovered initial condition)
u_ana_int, v_ana_int, p_ana_int = unpack(result.x)

## 7. Convergence Plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.semilogy(cost_history, "b-", linewidth=2)
ax1.axhline(float(J_at_truth), color="g", linestyle="--", label="Cost at truth")
ax1.set_xlabel("Function evaluation", fontsize=13)
ax1.set_ylabel("Cost J", fontsize=13)
ax1.set_title("Cost function convergence", fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

ax2.semilogy(grad_norm_history, "r-", linewidth=2)
ax2.set_xlabel("Function evaluation", fontsize=13)
ax2.set_ylabel("|∇J|", fontsize=13)
ax2.set_title("Gradient norm convergence", fontsize=14)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Visualize the Recovery

Compare truth, background, and analysis (recovered) initial conditions.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# --- Pressure field p ---
vmin_p = min(float(p_true_int.min()), float(p_bg_int.min()), float(p_ana_int.min()))
vmax_p = max(float(p_true_int.max()), float(p_bg_int.max()), float(p_ana_int.max()))

im = axes[0, 0].imshow(
    np.array(p_true_int), cmap="viridis", origin="lower", vmin=vmin_p, vmax=vmax_p
)
axes[0, 0].set_title("Truth p₀", fontsize=13)
plt.colorbar(im, ax=axes[0, 0], shrink=0.8)

im = axes[0, 1].imshow(np.array(p_bg_int), cmap="viridis", origin="lower", vmin=vmin_p, vmax=vmax_p)
axes[0, 1].set_title("Background p₀", fontsize=13)
plt.colorbar(im, ax=axes[0, 1], shrink=0.8)

im = axes[0, 2].imshow(
    np.array(p_ana_int), cmap="viridis", origin="lower", vmin=vmin_p, vmax=vmax_p
)
axes[0, 2].set_title("Analysis (recovered) p₀", fontsize=13)
plt.colorbar(im, ax=axes[0, 2], shrink=0.8)

# --- Error fields ---
err_bg = np.array(p_bg_int - p_true_int)
err_ana = np.array(p_ana_int - p_true_int)
vmax_err = max(np.abs(err_bg).max(), np.abs(err_ana).max())

axes[1, 0].set_visible(False)

im = axes[1, 1].imshow(err_bg, cmap="RdBu_r", origin="lower", vmin=-vmax_err, vmax=vmax_err)
axes[1, 1].set_title(f"Background error (RMS={np.sqrt(np.mean(err_bg**2)):.2f})", fontsize=13)
plt.colorbar(im, ax=axes[1, 1], shrink=0.8)

im = axes[1, 2].imshow(err_ana, cmap="RdBu_r", origin="lower", vmin=-vmax_err, vmax=vmax_err)
axes[1, 2].set_title(f"Analysis error (RMS={np.sqrt(np.mean(err_ana**2)):.2f})", fontsize=13)
plt.colorbar(im, ax=axes[1, 2], shrink=0.8)

fig.suptitle("4D-Var Initial Condition Recovery — Pressure Field", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

# Quantitative metrics
rms_bg = float(jnp.sqrt(jnp.mean((p_bg_int - p_true_int) ** 2)))
rms_ana = float(jnp.sqrt(jnp.mean((p_ana_int - p_true_int) ** 2)))
print(f"\nPressure field recovery:")
print(f"  Background RMS error: {rms_bg:.4f}")
print(f"  Analysis RMS error:   {rms_ana:.4f}")
print(f"  Error reduction:      {rms_bg / rms_ana:.1f}x")

In [ ]:
# Also show u and v recovery
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for row, (field_true, field_bg, field_ana, name) in enumerate(
    [
        (u_true_int, u_bg_int, u_ana_int, "u"),
        (v_true_int, v_bg_int, v_ana_int, "v"),
    ]
):
    err_bg = np.array(field_bg - field_true)
    err_ana = np.array(field_ana - field_true)
    vmax = max(np.abs(err_bg).max(), np.abs(err_ana).max())

    im = axes[row, 0].imshow(np.array(field_true), cmap="RdBu_r", origin="lower")
    axes[row, 0].set_title(f"Truth {name}₀", fontsize=13)
    plt.colorbar(im, ax=axes[row, 0], shrink=0.8)

    im = axes[row, 1].imshow(err_bg, cmap="RdBu_r", origin="lower", vmin=-vmax, vmax=vmax)
    rms = np.sqrt(np.mean(err_bg**2))
    axes[row, 1].set_title(f"Background {name}₀ error (RMS={rms:.4f})", fontsize=13)
    plt.colorbar(im, ax=axes[row, 1], shrink=0.8)

    im = axes[row, 2].imshow(err_ana, cmap="RdBu_r", origin="lower", vmin=-vmax, vmax=vmax)
    rms = np.sqrt(np.mean(err_ana**2))
    axes[row, 2].set_title(f"Analysis {name}₀ error (RMS={rms:.4f})", fontsize=13)
    plt.colorbar(im, ax=axes[row, 2], shrink=0.8)

fig.suptitle("4D-Var Recovery — Velocity Fields (u, v)", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

# Print all RMS errors
for field_true, field_bg, field_ana, name in [
    (u_true_int, u_bg_int, u_ana_int, "u"),
    (v_true_int, v_bg_int, v_ana_int, "v"),
    (p_true_int, p_bg_int, p_ana_int, "p"),
]:
    rms_bg = float(jnp.sqrt(jnp.mean((field_bg - field_true) ** 2)))
    rms_ana = float(jnp.sqrt(jnp.mean((field_ana - field_true) ** 2)))
    reduction = rms_bg / rms_ana if rms_ana > 0 else float("inf")
    print(
        f"{name}: background RMS = {rms_bg:.6f}, analysis RMS = {rms_ana:.6f}, reduction = {reduction:.1f}x"
    )

## 9. Forecast Verification

Does the recovered initial condition also produce a better *forecast*?
Let's run the model forward beyond the assimilation window and compare forecast errors
against truth. We extend the forecast to 2x the window length to test whether the
improved initial condition retains its advantage.

In [ ]:
# Run forecasts from truth, background, and analysis
# Forecast for 2x the assimilation window to test persistence of improvement
N_FORECAST = 2 * N_STEPS

print(f"Running {N_FORECAST}-step ({N_FORECAST * dt / 3600:.1f}h) forecasts...")
u_f_true, v_f_true, p_f_true = forward_model(u_true_int, v_true_int, p_true_int, N_FORECAST)
u_f_bg, v_f_bg, p_f_bg = forward_model(u_bg_int, v_bg_int, p_bg_int, N_FORECAST)
u_f_ana, v_f_ana, p_f_ana = forward_model(u_ana_int, v_ana_int, p_ana_int, N_FORECAST)
p_f_ana.block_until_ready()

# Compare forecast errors (interior only)
p_true_fc = p_f_true[1:-1, 1:-1]
err_bg_fc = np.array(p_f_bg[1:-1, 1:-1] - p_true_fc)
err_ana_fc = np.array(p_f_ana[1:-1, 1:-1] - p_true_fc)

rms_bg_fc = float(np.sqrt(np.mean(err_bg_fc**2)))
rms_ana_fc = float(np.sqrt(np.mean(err_ana_fc**2)))

# Plot forecast errors, not absolute fields
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

vmax_err = max(np.abs(err_bg_fc).max(), np.abs(err_ana_fc).max())

im = axes[0].imshow(err_bg_fc, cmap="RdBu_r", origin="lower", vmin=-vmax_err, vmax=vmax_err)
axes[0].set_title(f"Background forecast error\n(RMS = {rms_bg_fc:.2f})", fontsize=13)
plt.colorbar(im, ax=axes[0], shrink=0.8)

im = axes[1].imshow(err_ana_fc, cmap="RdBu_r", origin="lower", vmin=-vmax_err, vmax=vmax_err)
axes[1].set_title(f"Analysis forecast error\n(RMS = {rms_ana_fc:.2f})", fontsize=13)
plt.colorbar(im, ax=axes[1], shrink=0.8)

fig.suptitle(
    f"Forecast error at t = {N_FORECAST * dt / 3600:.1f}h (pressure field, forecast − truth)",
    fontsize=14,
    y=1.02,
)
plt.tight_layout()
plt.show()

print(f"\nForecast RMS error (pressure at t = {N_FORECAST * dt / 3600:.1f}h):")
print(f"  From background IC: {rms_bg_fc:.4f}")
print(f"  From analysis IC:   {rms_ana_fc:.4f}")
print(f"  Improvement:        {rms_bg_fc / rms_ana_fc:.1f}x")

## 10. AD Gradient vs Finite Differences

To highlight the advantage of AD: let's compare the cost of computing the gradient
via AD (one reverse pass) vs finite differences (one forward pass per control variable).

In [ ]:
# Time one AD gradient evaluation
t0 = perf_counter()
for _ in range(5):
    val, grads = cost_and_grad_fn(u_bg_int, v_bg_int, p_bg_int)
    grads[2].block_until_ready()
t_ad = (perf_counter() - t0) / 5

# Time one forward model evaluation (for finite difference comparison)
forward_jit = jax.jit(lambda u, v, p: cost_4dvar(u, v, p))
_ = forward_jit(u_bg_int, v_bg_int, p_bg_int)  # warmup

t0 = perf_counter()
for _ in range(5):
    val = forward_jit(u_bg_int, v_bg_int, p_bg_int)
    val.block_until_ready()
t_forward = (perf_counter() - t0) / 5

n_control = 3 * M * N  # total control variables
t_fd_estimated = n_control * t_forward  # finite differences need one eval per variable

print(f"Control vector dimension: {n_control}")
print(f"\nGradient computation time:")
print(f"  AD (one reverse pass):          {t_ad:.4f}s")
print(f"  Forward model evaluation:       {t_forward:.4f}s")
print(f"  Finite differences (estimated): {t_fd_estimated:.2f}s ({n_control} forward evals)")
print(f"\nAD speedup over finite differences: {t_fd_estimated / t_ad:.0f}x")
print(f"AD overhead over single forward:    {t_ad / t_forward:.1f}x")

## Summary

**What we demonstrated:**

1. **4D-Var data assimilation** successfully recovers the true initial conditions of a
   shallow water flow from **sparse** (50% coverage), noisy observations of all three
   fields (u, v, p) at 5 times within a 100-timestep (2.5 hour) assimilation window.

2. The gradient of the 4D-Var cost function is computed automatically by **JAX's reverse-mode AD**
   through the full time integration — no manual adjoint coding required.

3. **L-BFGS optimization** reduces the initial condition error significantly, demonstrating
   that the model dynamics propagate observation information to unobserved grid points.

4. The recovered initial condition produces a better **forecast** than the background.

5. AD-computed gradients are **orders of magnitude faster** than finite differences
   for high-dimensional control spaces.

**Key enabler:** The `swm_array_api.py` implementation uses a functional, mutation-free style
that is directly compatible with JAX's AD — no code modifications were needed.

**Simplifications relative to operational systems:** This demo uses diagonal error covariances
($B$ and $R$ are scalar multiples of identity per field), a small grid (16x16), and a
single outer loop. Operational 4D-Var systems (e.g., ECMWF's IFS) use spatially correlated
$B$ via diffusion operators, incremental inner/outer loop formulations at multiple resolutions,
and regularized linearized physics — but the core principle of adjoint-based optimization
of initial conditions is the same. JAX's automatic differentiation replaces the manual
adjoint coding that those systems require.